# 🚀 INFERNT — CUDA LLM Inference Engine

A ground-up CUDA/C++ inference engine for LLaMA-family architectures.

**⚠️ IMPORTANT:** Set your runtime to GPU before running: **Runtime → Change runtime type → T4 GPU**

## 1. Verify GPU & CUDA Environment

In [ ]:
# Verify GPU is available and check CUDA version
!nvidia-smi
print()
!nvcc --version
print()

# Show initial GPU memory state
!nvidia-smi --query-gpu=name,memory.used,memory.free,memory.total --format=csv,noheader

## 2. Clone & Build INFERNT

In [ ]:
%%bash
set -e

# Clone the repository (skip if already present)
if [ ! -d "/content/INFERNT" ]; then
    git clone https://github.com/Mahendra1706/INFERNT.git /content/INFERNT
fi

cd /content/INFERNT

# Clean build
rm -rf build
mkdir -p build
cd build

# Configure and build
cmake .. -DCMAKE_BUILD_TYPE=Release
make -j$(nproc)

echo ""
echo "✅ Build successful!"
echo "Binary: /content/INFERNT/build/bin/infernt"
ls -la /content/INFERNT/build/bin/infernt

## 3. Run Test Suite (Phases 1-18)

In [ ]:
%%bash
# Run the built-in test suite (no model needed, uses synthetic tiny tensors)
/content/INFERNT/build/bin/infernt

## 4. Convert TinyLlama Checkpoint to INFERNT Format

This step downloads TinyLlama-1.1B from Hugging Face and converts it to INFERNT's binary format.
**This will use ~4.5 GB of CPU RAM for conversion.**

In [ ]:
# Install dependencies for the converter
!pip install -q transformers torch accelerate sentencepiece protobuf

# Convert the checkpoint
!python3 /content/INFERNT/tools/convert_checkpoint.py \
    --model "TinyLlama/TinyLlama-1.1B-Chat-v1.0" \
    --output_dir /content/model_checkpoint

# Verify output files
print("\n📁 Checkpoint files:")
!ls -lh /content/model_checkpoint/

## 5. Run Inference with TinyLlama 🧠

This loads the full model onto the GPU and generates text. Watch the GPU memory usage!

In [ ]:
# Show GPU memory BEFORE loading the model
print("GPU memory BEFORE model load:")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader
print()

In [ ]:
%%bash
# Run inference with the converted model
/content/INFERNT/build/bin/infernt \
    --model /content/model_checkpoint \
    --prompt "Once upon a time, in a high-performance CUDA world," \
    --max-tokens 64 \
    --temp 0.0

In [ ]:
# Show GPU memory AFTER model load (should be ~4.4 GB used for TinyLlama-1.1B FP32)
print("GPU memory AFTER model load:")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader

## 6. Try Different Prompts

In [ ]:
%%bash
# Try another prompt with temperature sampling
/content/INFERNT/build/bin/infernt \
    --model /content/model_checkpoint \
    --prompt "The future of artificial intelligence" \
    --max-tokens 100 \
    --temp 0.7